# Curs 2 — Ecosistemul de modele

Scopul acestui notebook: testăm **2-3 modele diferite** pe același input și alegem modelul potrivit pentru proiect.

Vom folosi:
1. **Gemini** — providerul principal, prin cheia obținută din Google AI Studio.
2. **OpenRouter** — provider alternativ, util pentru comparație și backup când Gemini are limite de quota.
## OpenRouter — de unde luăm cheia
1. Intră pe https://openrouter.ai/
2. Creează cont sau autentifică-te.
3. Mergi la **Keys**.
4. Creează un nou API key.
5. Copiază cheia în fișierul `.env`:
```env
OPENROUTER_API_KEY=pune-cheia-ta-aici
---

In [19]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

## 1. Configurare — mai multe modele

In [31]:
MODELE = [
    ("deepseek", "deepseek-chat", "DeepSeek Chat"),
    ("openrouter", "qwen/qwen-2.5-7b-instruct:free", "Qwen 2.5 7B Free"),
    ("openrouter", "meta-llama/llama-3.1-8b-instruct:free", "Llama 3.1 8B Free"),
]

print("Modele pregătite:", [nume for _, _, nume in MODELE])

Modele pregătite: ['DeepSeek Chat', 'Qwen 2.5 7B Free', 'Llama 3.1 8B Free']


In [32]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

BASE_URLS = {
    "deepseek": "https://api.deepseek.com",
    "openrouter": "https://openrouter.ai/api/v1",
    "grok": "https://api.x.ai/v1",
}

API_KEYS = {
    "deepseek": os.getenv("DEEPSEEK_API_KEY"),
    "openrouter": os.getenv("OPENROUTER_API_KEY"),
    "grok": os.getenv("GROK_API_KEY"),
}

def make_client(provider):
    return OpenAI(
        api_key=API_KEYS[provider],
        base_url=BASE_URLS[provider]
    )

MODELE = [
    ("deepseek", "deepseek-chat", "DeepSeek Chat"),
    ("openrouter", "qwen/qwen-2.5-7b-instruct:free", "Qwen 2.5 7B Free"),
    ("openrouter", "meta-llama/llama-3.1-8b-instruct:free", "Llama 3.1 8B Free"),
    ("grok", "grok-3-mini", "Grok 3 Mini"),
]

print("Modele pregătite:", [nume for _, _, nume in MODELE])

Modele pregătite: ['DeepSeek Chat', 'Qwen 2.5 7B Free', 'Llama 3.1 8B Free', 'Grok 3 Mini']


## 2. Funcție helper — trimitem același prompt la orice model

În loc să scriem același cod de 3 ori, facem o funcție.

In [33]:
# varianta minimala pentru toate modelele

# fara functie
client = make_client("deepseek")

prompt = "Explică în 2 propoziții ce este un LLM."

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print("DeepSeek Chat:")
print(response.choices[0].message.content)


# cu functie
def ask(provider, model, prompt, temperature=0.1):

    client = make_client(provider)

    messages = [
        {"role": "user", "content": prompt}
    ]

    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=messages
    )

    return response.choices[0].message.content


# testare modele
MODELE = [
    ("deepseek", "deepseek-chat", "DeepSeek Chat"),
    ("openrouter", "qwen/qwen-2.5-7b-instruct:free", "Qwen 2.5 7B Free"),
    ("openrouter", "meta-llama/llama-3.1-8b-instruct:free", "Llama 3.1 8B Free"),
    ("grok", "grok-3-mini", "Grok 3 Mini"),
]

prompt_test = "Explică în 2 propoziții ce este un LLM."

for provider, model, nume in MODELE:

    print("\n" + "=" * 60)
    print(nume)

    try:
        raspuns = ask(
            provider=provider,
            model=model,
            prompt=prompt_test,
            temperature=0.1
        )

        print(raspuns)

    except Exception as e:
        print("[Eroare API:", e, "]")

DeepSeek Chat:
Un LLM (Model de Limbaj de mari dimensiuni) este un algoritm avansat de inteligență artificială, antrenat pe cantități masive de text, care învață să prezică și să genereze cuvinte pe baza contextului. Practic, funcționează ca un sistem extrem de sofisticat de completare a textului, capabil să creeze propoziții coerente și să răspundă la întrebări, deși nu posedă gândire sau conștiință reală.

DeepSeek Chat
Un LLM (Large Language Model) este un tip de inteligență artificială antrenată pe cantități uriașe de text pentru a înțelege și genera limbaj uman. Practic, funcționează ca un predictor avansat de cuvinte, capabil să creeze răspunsuri coerente, să traducă sau să rezume informații pe baza contextului primit.

Qwen 2.5 7B Free
[Eroare API: Error code: 404 - {'error': {'message': 'No endpoints found for qwen/qwen-2.5-7b-instruct:free.', 'code': 404}, 'user_id': 'user_3D5EKXDOPFu5da8XZ5hwLzDwKRw'} ]

Llama 3.1 8B Free
[Eroare API: Error code: 404 - {'error': {'message': '

In [34]:
from openai import RateLimitError, APIError, AuthenticationError
import json

def ask(provider, model, prompt, system=None, temperature=0.7, json_schema=None):
    """Trimite un prompt la model. Poate returna text simplu sau JSON structurat."""

    client = make_client(provider)

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    extra_args = {}

    if json_schema:
        extra_args["response_format"] = {
            "type": "json_schema",
            "json_schema": json_schema
        }

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            **extra_args
        )

        text = response.choices[0].message.content.strip()

        if json_schema:
            return json.loads(text)

        return text

    except RateLimitError:
        return f"[Eroare: quota/rate limit pentru modelul {model}.]"

    except AuthenticationError:
        return "[Eroare: API key invalidă sau lipsă. Verifică .env.]"

    except APIError as e:
        return f"[Eroare API: {e}]"

    except Exception as e:
        return f"[Eroare: {type(e).__name__} — {e}]"

## 3. Test 1 — Calitatea pe limba română

Testăm dacă modelele înțeleg și răspund corect în română.

In [35]:
PROMPT_RO = """
Rezumă în exact 2 propoziții scurte, în română, principalele schimbări din politica românească din ultimii 7 ani.
Maximum 100 de cuvinte.
Răspunde pe baza faptelor, fără opinii politice.
"""

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    raspuns = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT_RO,
        temperature=0.2
    )

    print(raspuns)


--- DeepSeek Chat ---
În ultimii 7 ani, România a trecut printr-o alternanță la guvernare între PSD și PNL, inclusiv un guvern tehnocrat și o coaliție PNL-USR, urmată de revenirea PSD la putere. De asemenea, s-au înregistrat modificări legislative majore în justiție și s-a accelerat procesul de aderare la Spațiul Schengen, finalizat parțial în martie 2024.

--- Qwen 2.5 7B Free ---
[Eroare API: Error code: 404 - {'error': {'message': 'No endpoints found for qwen/qwen-2.5-7b-instruct:free.', 'code': 404}, 'user_id': 'user_3D5EKXDOPFu5da8XZ5hwLzDwKRw'}]

--- Llama 3.1 8B Free ---
[Eroare API: Error code: 404 - {'error': {'message': 'No endpoints found for meta-llama/llama-3.1-8b-instruct:free.', 'code': 404}, 'user_id': 'user_3D5EKXDOPFu5da8XZ5hwLzDwKRw'}]

--- Grok 3 Mini ---
[Eroare API: Error code: 400 - {'code': 'Client specified an invalid argument', 'error': 'Incorrect API key provided: gs***G3. You can obtain an API key from https://console.x.ai.'}]


In [36]:
MODELE = [
    ("deepseek", "deepseek-chat", "DeepSeek Chat"),
]

print("Modele pregătite:", [nume for _, _, nume in MODELE])

Modele pregătite: ['DeepSeek Chat']


## 4. Test 2 — Urmează instrucțiunile din system prompt+ adnotare

Vedem dacă modelele respectă rolul dat prin `system`.

In [37]:
SYSTEM = """
Ești un asistent de cercetare care adnotează comentarii politice.
Răspunzi scurt, clar și nu inventezi informații.
"""

PROMPT = """
Analizează următorul comentariu politic:
"Toți politicienii fură, iar oamenii simpli plătesc nota. Nimeni nu mai ascultă poporul."

Răspunde în 4 linii:
Ton:
Emoție dominantă:
Țintă principală:
Populism: da/nu
"""

for provider, model, name in MODELE:
    print("\n---", name, "---")
    print(ask(
        provider=provider,
        model=model,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0
    ))


--- DeepSeek Chat ---
Ton: Cinic și acuzator.
Emoție dominantă: Frustrare și neîncredere.
Țintă principală: Clasa politică în ansamblu.
Populism: da


## 5. Test 3 — Output structurat (JSON)

Agenții noștri vor trebui să returneze date structurate.
Testăm dacă modelele pot produce JSON valid la cerere.

In [38]:
SCHEMA_ADNOTARE = {
    "name": "adnotare_comentariu_politic",
    "schema": {
        "type": "object",
        "properties": {
            "ton": {
                "type": "string",
                "enum": ["pozitiv", "negativ", "neutru"]
            },
            "emotie_dominanta": {
                "type": "string",
                "enum": ["furie", "frica", "speranta", "dezamagire", "ironie", "neutru"]
            },
            "tinta_principala": {
                "type": "string"
            },
            "populism": {
                "type": "boolean"
            },
            "explicatie_scurta": {
                "type": "string"
            }
        },
        "required": [
            "ton",
            "emotie_dominanta",
            "tinta_principala",
            "populism",
            "explicatie_scurta"
        ],
        "additionalProperties": False
    }
}

In [39]:
COMENTARIU = "Toți politicienii fură, iar oamenii simpli plătesc nota. Nimeni nu mai ascultă poporul."

SYSTEM = "Ești un asistent de cercetare care adnotează comentarii politice."

PROMPT = f"Adnotează următorul comentariu politic: {COMENTARIU}"

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    rezultat = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0.1,
        json_schema=SCHEMA_ADNOTARE
    )

    print(rezultat)


--- DeepSeek Chat ---
[Eroare API: Error code: 400 - {'error': {'message': 'This response_format type is unavailable now', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}]


## 6. Test 4 — Stabilitate la temperature diferite

Un model bun pentru agenți trebuie să fie **stabil** — același input, răspunsuri similare.
Testăm cu Gemini (poți schimba cu orice model).

In [40]:
PROMPT_STAB = """
Curtea Constituțională a anulat alegerile.
Explică în 2 propoziții ce poate însemna acest lucru pentru viața politică.
Răspunde neutru, fără opinii partizane.
"""

TEMPERATURI = [0.1, 0.7, 1.2]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE:
    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)

[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ DeepSeek Chat ]

temperature=0.1:
Anularea alegerilor de către Curtea Constituțională poate duce la o criză politică și la instabilitate instituțională, deoarece procesul electoral trebuie reluat, iar încrederea publicului în sistemul democratic poate fi afectată. De asemenea, aceasta poate genera dezbateri intense privind legalitatea și corectitudinea procedurilor electorale, influențând agenda politică și relațiile dintre partide.

temperature=0.7:
Anularea alegerilor de către Curtea Constituțională poate indica o criză de încredere în procesul electoral, generând incertitudine politică și întârzieri în formarea instituțiilor statului. Acest eveniment poate duce fie la organizarea unui nou scrutin, care să restabilească legalitatea, fie la escaladarea tensiunilor între actorii politici și contestarea legitimității deciziei.

temperature=1.2:
În cazul anulării alegerilor de către Curtea Constituțională, procesul elector

## 7. Alegerea modelului pentru proiect

Completați tabelul după testele de mai sus. Nu căutați „cel mai bun model” în general, ci modelul cel mai potrivit pentru proiectul vostru.
| Model | Răspunde bine în română? | Respectă instrucțiunile? | Merge pentru adnotare? | Are erori / quota? | Observație scurtă |
|---|---|---|---|---|---|
| Gemini 2.5 Flash Lite | erori quota|
| Deepseek Chat| da | da | da | nu  |
| Restul modelelor incercate au avut erori
## Decizie

**Model principal ales:** DeepSeek Chat  
**Model de rezervă:** niciunul  
**Temperature recomandată:** 0.1  

La testul de stabilitate, DeepSeek Chat a oferit răspunsuri coerente la toate cele trei temperaturi. La temperature=0.1, răspunsul este mai controlat și mai potrivit pentru adnotare, deoarece formulează clar efectele politice și instituționale. La temperature=0.7 și 1.2, răspunsurile rămân corecte, dar devin ușor mai variate ca formulare și includ interpretări mai largi. Pentru adnotare, aleg temperature=0.1 deoarece oferă rezultate mai stabile și mai consistente.

## 8. Configurația finală a proiectului

putem să copiem asta in core/config.py

In [17]:
# core/config.py
# Configurația modelului ales de echipă după testele din Cursul 2.
# Nu puneți chei API aici. Cheile rămân doar în fișierul local .env.
PROVIDER_PRINCIPAL = "gemini"
MODEL_PRINCIPAL = "gemini-2.5-flash-lite"
PROVIDER_FALLBACK = "openrouter"
MODEL_FALLBACK = "openrouter/free"
TEMPERATURE = 0.2

---

## Livrabile C2

Până la cursul următor:

- [ ] Notebook completat cu 2-3 modele testate
- [ ] Matricea de decizie completată cu observații reale
- [ ] README actualizat cu modelul ales și justificarea
- [ ] `.env` configurat cu cheia pentru modelul ales